LETTURA, VISUALIZZAZIONE E SALVATAGGIO DELLE IMMAGINI

- Gestione input/output: interazione puva con cv2
- Formati e flag: perchè non basta dire di carica immagine ma anche spiegare come interpretare
- Pipeline di batchprocessing automatizzate

Fondamenti di I/O con Open CV
Il passaggio dal file al buffer di memoria
Pensa al caricamente come ad un'operazione di spacchettamento. Un file jpg sul disco è come un pacco regalo ben sigillato e compresso, finchè non lo apri non sai cosa farci
Caricare l'immagine significa estrarre dei dati e distribuirli  ordinamente nella ram in un array np. Solo in quel momento i pixel diventano vivi e pronti per essere manipolati dai nostri algoritmi.

Quali sono gli strumenti che open cv ci mette a distribuzione

- cv2.imrtead() è la porta d'ingresso. se sbagliate il percorso del file non ho un errore ma null. 
- cv2.imshow() monitor di controllo. 
- cv2.waitkey imshow la finestra sparisce in un millisencondo se non uso waitkey, se imposto a 0 il programma si ferma fintanto che non viene premuto un tasto
- cv2.destroy: per liberare spazio

Il paradosso del colore in OpenCV
l'ordine dei colori è BGR, come se stesse leggendo un libro al contrario. Questo differisce dalla standard RGB usato dalla maggior parte delle altre librerie,
Le finestre di Open CV sono thread-dependement. Senza l'uso corretto di waitKey, la finestra potrebbe apparire come congelata o non rispondere agli eventi del sisetma.
Il rendering avviene mappando l'indice di riga sull'asse verticale e l'indice di colonna sull'asse orizzontale partendo dall'origine in alto a sinistra.
Si parte sempre nell'angolo in alto a sinistra, si scende per le righe e si va a destra per le colonne.

Quante momoria occupa?

Efficienza di Caricamento
non sottovalutare il peso delle immagini.
Mentre un file jpeg potrebbe posare poco, una volta caricato nella RAM occupa tutto il suo spazio reale.
La formula B=H W C è la formula per lo spazio in memoria
moltimplico le dimensioni per la prodondità di bit del tipo di dato.

Oltre le dimensioni dobbiamo decidere con quali mezzi leggere il file.
I file immagine non è solo un gruppo di pixel, contiene anche informazioni su come deve essere interpretato.
Non tutte le immagini vengono caricate allo stesso modo, Open CV fornisce dei flag che istruiscono l decoder su come interpretare i dati grezzi presenti nel file.
La scelta del formato di salvataggio (JPG o PNG) non influenza solo lo spazio su disco, ma anche l'integrita dei dati che verranno processati dalla rete neurale, influenza direttamente quanto la rete riuscirà ad imparare.

Flag di Caricamento
Istruire il decoder di Open CV
* IMREAD_COLOR: il flag predefinito. Carica l'immagine a 8 bit per canale rimuovendo eventuali canali di trasparenza esistenti. Mi da il colore anche se l'immagine è grigia
* IMREAD_GRAYSCALE: converte l'immagine in un singolo canale di intensità luminosa direttamente durante la fase di decodifa del file
* IMREAD_UNCHANGED: fondamentale per i file PNG, poichè mantiene il canale Alpha (trasparenza) restituendo un array a 4 canali (BGRA). Es. se devo allenare l'ai per rimuovere lo sfondo ho bisogno del cancale Alpha che rappresenta la trasparenza.
*IMREAD_ANYDEPTH: permette di caricare immagini con precisione superiore a 8 bit (es. TIFF a 16 bit) senza declassare la dinamica del segnale. Utilizzato per dati scientifici o medici, dove ho bisogno di tutta la precisoine del tensore originale

Ma una volta caricata l'immagine, come la salvo? (JPG o PNG)?

JPG vs PNG nel Deep Learning
Il formato JPG elimina informazioni ad alta frequenza per ridurre il peso. Questo può introdurre artefatti che distrurbano delle feature più fini.
Il JPG è un formato lossy, per risparmiare spazio inventa dei pixel o ne fonde altri, per l'occhio umano va bene ma per una rete neurale questi sono rumore
Il PNG conserva ogni singolo bit originale. E' il formato predefinito per la segmentazione medica o industruale dove ogni pixel ha un valore semantico preciso.
Il PNG lossness è la verità assoluta del dato, è più pesante ma è obbligatorio quando la precisione è fondamentale.
La funzione cv2.imwrite permette di specificare parametri di qualità o compressione tramite una lista di coppie chiave-valore specifiche per il formato scelto.

Come quantificare il risparmio di spazio.
Integrità del Segnale
Rapporto di Compressione (R)
R= size-raw/seze-file
Il rapporto di compressione è il termometro della nostra efficienza. Se R è molto alto, significa che abbiamo sascrificato molta informazione per risparmiare spazio. 
La scelta della qualità nel formato JPG bilancia il peso del dataset e la fedeltà del segnale. Un valore torppo basso genera rumore di quantizzazione. 
Il rapport di compression 'R' indica quanto il file è stato ridotto rispetto alla sua rappresentazione grezza non compressa nel buffer di memoria.

Bisogno travare il punto di equilibrio, un dataset troppo ampio rallenta il caricamento dai dischi ma un datase troppo compresso degrada la qualità del segnale.

Ora che sappiamo gestire una singola immagine passiamo alla produzione industriale

Automazion e Scripting Batch
Scalabilità del processamento visivo
Nel Deep Learning raramente lavoriamo su una singola immagine. Dobbiamo essere in grado di iterare su migliaia di file in modo robusto e veloce.
Combiniamo OpenCV con i moduli di sistema di Python per creare flussi di lavoro automatizzati che preparano i dati per l'addestramento.

Ma come possiamo esplorare il nostro HD con Python?

Scansione del File System
Gestione dei percorsi e iterazioni
* Modulo 'os' e 'glob': strumenti standard per listare i file contenuti in una directory e filtrare solo quelli con estensione specifiche. Ma nel 2026 lo standard per eccellenza è pathlib
* os.path.join(): metodo essenziale per costruire percorsi di file validi su diversi sistemi operativi (Windows, Linux, macOS)
* Loop Iterativo: la struttura fondamentale che carica un'immagine, applica una trasformazione e salva il risultato in una cartella di output.
* Filtro Estensioni: l'importanza di ignorare file nascosti o metadati di sistema (come .DS_Store) che causerebbero il crash del caricamento.

Oltre al filtro servono buone maniere di controllo del nostro codice

Best Practies di Automazione

* Verifica dell'esistenza: prima di salvare, è buona norma verificare che la cartella di destinazione esiste utilizzando os.makedirs con l'opzione exist_ok=True
* Logging dei progressi: durante il batch processing, stampare a video il nome del file corrente aiuta a identificare file corrotti che bloccano la pipeline.
* Efficienza temporale: evitare di visualizzare le immagini con cv2.imshow durante il ciclo di batch su migliaia di file, poichè il rendering a video rallenta drasticamente il processo.

Come misuriamo quanto è veloce la nostra catena di montaggio?

Metriche di Processamento
Calcolo del Throughput
Misurare la velocità di elaborazione è fondamentale per stimare i tempi di preparazione di grandi dataset prima della fase di training.
Il throughput (T) rappresenta il numero di immagini processate nell'unità di tempo e dipende sia dalla velocità del disco che dalla complessità dell'algoritmo.
Se T è basso il disco è troppo lento e l'algoritmo è troppo pesante.


In [3]:
import os

# Best practice 2026: Configurazione del backend di Keras 3 prima dell'importazione
# Utilizziamo PyTorch come motore di calcolo per la sua flessibilità nel Deep Learning
os.environ["KERAS_BACKEND"] = "torch"

import cv2
import numpy as np
import keras
from pathlib import Path
from typing import Optional
import requests

def download_demo_images(target_dir: str, count: int = 2):
    """
    Scarica immagini casuali da internet per popolare la cartella di test.
    """
    path = Path(target_dir)
    path.mkdir(parents=True, exist_ok=True)
    
    print(f"[*] Download di {count} immagini di test in corso...")
    
    for i in range(count):
        # Utilizziamo Picsum per ottenere immagini casuali ogni volta
        url = f"https://picsum.photos/640/480?random={i}"
        try:
            response = requests.get(url, timeout=10)
            if response.status_code == 200:
                file_path = path / f"test_image_{i}.jpg"
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                print(f"[+] Scaricata: {file_path.name}")
        except Exception as e:
            print(f"[-] Errore durante il download: {e}")


def process_image_pipeline(image_path: Path, output_folder: Path):
    """
    Esegue il caricamento, una trasformazione base e il salvataggio.
    Include concetti di gestione memoria e flag di decodifica.
    """
    # 1. LETTURA (cv2.imread)
    # Teoricamente: imread decodifica i flussi di byte compressi (JPG/PNG) in array NumPy.
    # Flag IMREAD_COLOR: Carica a 8-bit per canale (0-255), scarta la trasparenza.
    img = cv2.imread(str(image_path), cv2.IMREAD_COLOR)

    # Controllo robustezza: imread non solleva eccezioni ma restituisce None se il path è errato
    if img is None:
        print(f"[-] Errore: Impossibile caricare {image_path.name}. File corrotto o percorso non valido.")
        return

    # 2. VISUALIZZAZIONE (Solo per debug, non consigliata in batch massivi)
    # Nota: Il nome della finestra 'Anteprima' serve come identificatore nel Window Manager
    cv2.imshow('Anteprima', img)
    
    # waitKey(500) blocca l'esecuzione per 500ms permettendo al sistema operativo di renderizzare la finestra.
    # Se premuto un tasto prima dei 500ms, restituisce il codice ASCII del tasto.
    cv2.waitKey(500)

    # 3. TRASFORMAZIONE (Esempio: Conversione in scala di grigi)
    # La formula della luminanza pesata è: Y = 0.299*R + 0.587*G + 0.114*B
    gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 4. SALVATAGGIO (cv2.imwrite)
    # Best Practice: Usiamo il formato PNG per evitare artefatti di compressione durante lo sviluppo
    # o JPG con parametri di qualità definiti per risparmiare spazio nel dataset finale.
    target_path = output_folder / f"processed_{image_path.stem}.jpg"
    
    # Specifichiamo la qualità JPG (da 0 a 100). Default è 95. 
    # IMWRITE_JPEG_QUALITY influenza il quantizzatore nella trasformata discreta del coseno (DCT).
    success = cv2.imwrite(str(target_path), gray_img, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    
    if success:
        print(f"[+] Salvato: {target_path.name}")

def batch_processor(source_dir: str, target_dir: str):
    """
    Scansiona una cartella e processa tutte le immagini in modo automatizzato.
    """
    # Pathlib permette una gestione dei percorsi cross-platform (Windows/Linux/Mac)
    src_path = Path(source_dir)
    dst_path = Path(target_dir)

    # Crea la cartella di destinazione se non esiste (Best practice: exist_ok=True)
    dst_path.mkdir(parents=True, exist_ok=True)

    # Definiamo le estensioni supportate per evitare file di sistema (es. .DS_Store o .json)
    valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

    # Iterazione efficiente sui file della directory
    print(f"[*] Inizio processamento nella cartella: {src_path}")
    for file in src_path.iterdir():
        if file.suffix.lower() in valid_extensions:
            process_image_pipeline(file, dst_path)
    
    # Pulizia finale: chiude tutte le finestre di sistema aperte da OpenCV
    cv2.destroyAllWindows()
    print("[*] Batch processing completato.")

if __name__ == "__main__":
    
    # Definizione cartelle
    raw_folder = "dataset_raw"
    processed_folder = "dataset_processed"

    # 1. Scarichiamo le immagini (Nuova funzione)
    download_demo_images(raw_folder, count=2)

    # 2. Eseguiamo il processamento batch originale
    batch_processor(raw_folder, processed_folder)

[*] Download di 2 immagini di test in corso...
[+] Scaricata: test_image_0.jpg
[+] Scaricata: test_image_1.jpg
[*] Inizio processamento nella cartella: dataset_raw
[+] Salvato: processed_test_image_0.jpg
[+] Salvato: processed_test_image_1.jpg
[*] Batch processing completato.
